# 1. Caricamento del dataset

In questo notebook viene valutato il classificatore Naive Bayes progettato nel Task 2 sul dataset `training.csv`.

L'obiettivo è verificare la capacità del modello di generalizzare su un insieme di dati significativamente più ampio rispetto al file `manuale.csv`.

Il dataset utilizzato contiene 50.000 osservazioni estratte in modo stratificato dal dataset originale.

In [5]:
import pandas as pd

training_df = pd.read_csv("../data/training.csv")

training_df.head()

,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,0,1,1,25,1,0,0,1,1,1,...,0,2,0,0,0,0,7,5,8,0
1,0,1,1,29,0,0,0,1,1,1,...,0,2,0,0,0,0,10,5,6,0
2,0,0,1,25,0,0,0,1,0,1,...,0,2,0,0,0,1,12,4,6,0
3,1,0,1,28,0,0,0,0,0,0,...,0,2,0,0,0,0,8,6,5,1
4,0,1,1,29,1,0,0,1,0,1,...,0,3,5,0,0,1,6,4,8,0


# 2. Preparazione dei dati

Nel Task 2 le variabili BMI e Age sono state discretizzate per semplificare il calcolo delle probabilità condizionate del classificatore Naive Bayes.

Per garantire coerenza tra fase di progettazione e fase di valutazione, le stesse trasformazioni vengono applicate anche al dataset `training.csv`.

In [12]:
# Discretizzazione del BMI

training_df["BMI_Category"] = pd.cut(
    training_df["BMI"],
    bins=[0, 25, 30, 100],
    labels=["Normal", "Overweight", "Obese"],
    right=False
)

training_df[["BMI", "BMI_Category"]]

,BMI,BMI_Category
0,25,Overweight
1,29,Overweight
2,25,Overweight
3,28,Overweight
4,29,Overweight
...,...,...
49995,26,Overweight
49996,26,Overweight
49997,23,Normal
49998,36,Obese


In [13]:
# Tabella di contingenza per BMI_Categorry

pd.crosstab(
    training_df["BMI_Category"],
    training_df["Diabetes_binary"]
)

Diabetes_binary,0,1
BMI_Category,,
Normal,13354,817
Overweight,16429,2087
Obese,13250,4063


In [11]:
# Discretizzazione del Age

training_df["Age_Category"] = pd.cut(
    training_df["Age"],
    bins=[0, 5, 9, 12, 14],
    labels=["Bambino", "Giovane", "Adulto", "Anziano"],
    right=False
)

training_df[["Age", "Age_Category"]]

,Age,Age_Category
0,7,Giovane
1,10,Adulto
2,12,Anziano
3,8,Giovane
4,6,Giovane
...,...,...
49995,3,Bambino
49996,13,Anziano
49997,13,Anziano
49998,6,Giovane


In [14]:
# Tabella di contingenza per Age_Categorry

pd.crosstab(
    training_df["Age_Category"],
    training_df["Diabetes_binary"]
)

Diabetes_binary,0,1
Age_Category,,
Bambino,7318,230
Giovane,16368,1974
Adulto,14063,3449
Anziano,5284,1314


# 3. Richiamo del classificatore Naive Bayes

In questa sezione riportiamo le probabilità a priori e le probabilità condizionate calcolate nel Task 2

In [15]:
# Struttura dati con le probabilità a priori

priors = {
    0: 0.5,
    1: 0.5
}

In [16]:
# Struttura dati con tutte le probabilità condizionate calcolate precedentemente

probabilities = {
    "HighBP": {
        0: {0: 4/7, 1: 3/7},
        1: {0: 3/7, 1: 4/7}
    },

    "HighChol": {
        0: {0: 6/7, 1: 1/7},
        1: {0: 2/7, 1: 5/7}
    },

    "BMI_Category": {
        0: {
            "Normal": 5/10,
            "Overweight": 4/10,
            "Obese": 4/10
        },
        1: {
            "Normal": 2/10,
            "Overweight": 4/10,
            "Obese": 4/10
        }
    },

    "Age_Category": {
        0: {
            "Bambino": 3/11,
            "Giovane": 2/11,
            "Adulto": 3/11,
            "Anziano": 3/11
        },
        1: {
            "Bambino": 1/11,
            "Giovane": 2/11,
            "Adulto": 6/11,
            "Anziano": 2/11
        }
    },

    "PhysActivity": {
        0: {0: 3/7, 1: 4/7},
        1: {0: 2/7, 1: 5/7}
    }
}

# 4. Applicazione del classificatore

Dopo aver ricostruito le probabilità a priori e condizionate ottenute nel Task 2, il classificatore Naive Bayes viene applicato alle osservazioni presenti nel dataset `training.csv`.


Per ogni osservazione vengono calcolati i punteggi associati alle due classi possibili (`Diabetes_binary = 0` e `Diabetes_binary = 1`):

- Score(Y=0 | X) = P(Y=0) × ∏ P(Xi | Y=0)
- Score(Y=1 | X) = P(Y=1) × ∏ P(Xi | Y=1)

La classe assegnata è quella con punteggio maggiore.

Si riporta la funzione `predict_naive_bayes` implementata nel Task 2. 

In [61]:
def predict_naive_bayes(row):

    # Probabilità a priori
    score_0 = priors[0]
    score_1 = priors[1]

    # HighBP
    score_0 *= probabilities["HighBP"][0][row["HighBP"]]
    score_1 *= probabilities["HighBP"][1][row["HighBP"]]

    # HighChol
    score_0 *= probabilities["HighChol"][0][row["HighChol"]]
    score_1 *= probabilities["HighChol"][1][row["HighChol"]]

    # BMI_Category
    score_0 *= probabilities["BMI_Category"][0][row["BMI_Category"]]
    score_1 *= probabilities["BMI_Category"][1][row["BMI_Category"]]

    # Age_Category
    score_0 *= probabilities["Age_Category"][0][row["Age_Category"]]
    score_1 *= probabilities["Age_Category"][1][row["Age_Category"]]

    # PhysActivity
    score_0 *= probabilities["PhysActivity"][0][row["PhysActivity"]]
    score_1 *= probabilities["PhysActivity"][1][row["PhysActivity"]]

    # Predizione finale
    prediction = 0 if score_0 > score_1 else 1

    return prediction, score_0, score_1


## Predizione del dataset

Il classificatore viene applicato a tutte le osservazioni del dataset di training.

In [31]:
training_df["Predicted"] = training_df.apply(
    lambda row: predict_naive_bayes(row)[0],
    axis=1
)

training_df[
    [
        "Diabetes_binary",
        "Predicted"
    ]
]

,Diabetes_binary,Predicted
0,0,1
1,0,1
2,0,0
3,1,0
4,0,1
...,...,...
49995,0,0
49996,0,1
49997,1,0
49998,1,0


In [32]:
predictions = training_df.apply(
    lambda row: predict_naive_bayes(row)[0],
    axis=1
)

predictions.value_counts()

0    27190
1    22810
Name: count, dtype: int64

# 5. Valutazione delle prestazioni

Dopo aver applicato il classificatore Naive Bayes al dataset di training, vengono calcolate le principali metriche di valutazione.

Le prestazioni vengono analizzate mediante:

- Confusion Matrix;
- Accuracy;
- Precision;
- Recall;
- F1-Score.

Tali metriche consentono di valutare il comportamento del classificatore su un dataset significativamente più ampio rispetto a quello utilizzato per la progettazione manuale.

# 5.1 Costruzione della tabella dei risultati

In [33]:
results_df = pd.DataFrame({
    "Actual": training_df["Diabetes_binary"],
    "Predicted": predictions
})

results_df.head()

,Actual,Predicted
0,0,1
1,0,1
2,0,0
3,1,0
4,0,1


# 5.2 Calcolo manuale della Confusion Matrix

In [34]:
tp = len(
    training_df[(training_df["Diabetes_binary"] == 1) & (training_df["Predicted"] == 1)]
)

tn = len(
    training_df[ (training_df["Diabetes_binary"] == 0) & (training_df["Predicted"] == 0)]
)

fp = len(
    training_df[ (training_df["Diabetes_binary"] == 0) & (training_df["Predicted"] == 1)]
)

fn = len(
    training_df[ (training_df["Diabetes_binary"] == 1) & (training_df["Predicted"] == 0)]
)

print("TP =", tp)
print("TN =", tn)
print("FP =", fp)
print("FN =", fn)

# Visualizzazione della matrice

confusion_matrix_df = pd.DataFrame(
    [
        [tn, fp],
        [fn, tp]
    ],
    columns=["Predicted 0", "Predicted 1"],
    index=["Actual 0", "Actual 1"]
)

confusion_matrix_df

TP = 5083
TN = 25306
FP = 17727
FN = 1884


,Predicted 0,Predicted 1
Actual 0,25306,17727
Actual 1,1884,5083


## Accuracy

L'accuracy misura la percentuale di osservazioni classificate correttamente dal modello.

La formula utilizzata è:

                Accuracy = (TP + TN) / (TP + TN + FP + FN)

dove:

- TP = True Positive
- TN = True Negative
- FP = False Positive
- FN = False Negative

In [36]:
accuracy = (tp + tn) / (tp + tn + fp + fn)

print("Accuracy: " , accuracy)

Accuracy:  0.60778


### Precision

La precision misura la percentuale di osservazioni classificate come positive che sono effettivamente positive.

La formula è:

            Precision = TP / (TP + FP)

Una precision elevata indica che il modello produce pochi falsi positivi, ovvero assegna la classe positiva solo quando è abastanza sicuro.

In [38]:
precision = tp / (tp + fp)

print("Precision: ", precision)

Precision:  0.222840859272249


### Recall

La recall misura la capacità del classificatore di individuare correttamente le osservazioni positive.

La formula è:

            Recall = TP / (TP + FN)

Una recall elevata indica che il modello perde pochi casi positivi, ovvero produce pochi falsi positivi.

In [39]:
recall = tp / (tp + fn)

print("Recall: ", recall)

Recall:  0.7295823166355677


### F1-Score

L'F1-Score rappresenta la media armonica tra precision e recall.

La formula è:

            F1 = 2 * (Precision * Recall) / (Precision + Recall)

Questa metrica è particolarmente utile quando si desidera trovare un compromesso tra precision e recall.

In [40]:
f1 = 2 * precision * recall / (precision + recall)

print("F1-Score: ", f1)

F1-Score:  0.3414044396682003


# 6. Analisi critica dei risultati

Il classificatore Naive Bayes progettato manualmente è stato applicato al dataset `training.csv`, contenente 50.000 osservazioni.

Le metriche ottenute sono:

| Metrica  | Valore  |
|----------|---------|
| Accuracy | 60.78%  |
| Precision| 22.28%  |
| Recall.  | 72.96%  |
| F1-Score | 34.14%  |

L'accuracy risulta moderata, ma non rappresenta da sola una misura affidabile delle prestazioni del classificatore, poiché il dataset presenta un forte sbilanciamento tra le classi.

La precision è relativamente bassa (22.28%), indicando che il classificatore tende a produrre molti falsi positivi. In altre parole, numerosi soggetti vengono classificati come diabetici pur non appartenendo realmente a tale categoria.

La recall risulta invece elevata (72.96%), mostrando una buona capacità del modello di individuare i soggetti realmente diabetici.

Dal punto di vista applicativo, specialmente in ambito sanitario, una recall elevata può essere considerata preferibile rispetto ad una precision elevata, poiché la mancata individuazione di un soggetto malato può avere conseguenze più gravi rispetto alla classificazione errata di un soggetto sano.

L'F1-score, pari al 34.14%, evidenzia tuttavia che il compromesso tra precision e recall non è ottimale.

Le limitazioni del modello sono probabilmente dovute al fatto che le probabilità sono state stimate utilizzando un dataset molto piccolo (`manuale.csv`), composto da sole 14 osservazioni. Tale campione non è sufficientemente rappresentativo della distribuzione reale presente nel dataset completo.

# 7. Tentativo di ottimizzazione del classificatore

Le probabilità utilizzate dal classificatore Naive Bayes sono state inizialmente stimate sul dataset `manuale.csv`, composto da sole 14 osservazioni.

Per verificare se una maggiore quantità di dati consenta di ottenere stime più affidabili delle probabilità condizionate, viene costruita una seconda versione del classificatore utilizzando un sottoinsieme più ampio del dataset `training.csv`.

L'obiettivo è confrontare le prestazioni del modello originale con quelle ottenute dopo la ricalibrazione delle probabilità.

In [46]:
from sklearn.model_selection import train_test_split

sample_df, _ = train_test_split(
    training_df,
    train_size=1000,
    stratify=training_df["Diabetes_binary"],
    random_state=42
)

sample_df.shape



(1000, 25)

In [47]:
sample_df.columns

Index(['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke',
       'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
       'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income',
       'Diabetes_binary', 'BMI_Category', 'Age_Category', 'Predicted'],
      dtype='object')

In [48]:
# Elimino colonna `Predicted` creata durante il Task 4

sample_df = sample_df.drop(
    columns=["Predicted"],
    errors="ignore"
)

sample_df.columns

Index(['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke',
       'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
       'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income',
       'Diabetes_binary', 'BMI_Category', 'Age_Category'],
      dtype='object')

## Ricalcolo delle probabilità

Per migliorare la stima delle probabilità utilizzate dal classificatore Naive Bayes, viene utilizzato un campione stratificato di 1000 osservazioni estratto dal dataset di training.

Le probabilità a priori vengono ricalcolate applicando lo stesso approccio utilizzato nel Task 2.

# Ricalcolo le nuove probabilità a priori

Nel Task 2 avevamo ottenuto:

- P(Y=0) = 0.5
- P(Y=1) = 0.5

In [51]:
n0 = len(
    sample_df[
        sample_df["Diabetes_binary"] == 0
    ]
)

n1 = len(
    sample_df[
        sample_df["Diabetes_binary"] == 1
    ]
)

prior_0 = n0 / len(sample_df)
prior_1 = n1 / len(sample_df)

print("P(Y=0) =", prior_0)
print("P(Y=1) =", prior_1)

P(Y=0) = 0.861
P(Y=1) = 0.139


## Ottimizzazione tramite aggiornamento delle probabilità a priori

Le nuove probabilità a priori risultano:

- P(Y=0)=0.861

- P(Y=1)=0.139

L'obiettivo è ridurre il numero di falsi positivi prodotti dal classificatore originale.

In [59]:
def predict_naive_bayes_v2(row):

    # Probabilità a priori
    score_0 = 0.861
    score_1 = 0.139

    # HighBP
    score_0 *= probabilities["HighBP"][0][row["HighBP"]]
    score_1 *= probabilities["HighBP"][1][row["HighBP"]]

    # HighChol
    score_0 *= probabilities["HighChol"][0][row["HighChol"]]
    score_1 *= probabilities["HighChol"][1][row["HighChol"]]

    # BMI_Category
    score_0 *= probabilities["BMI_Category"][0][row["BMI_Category"]]
    score_1 *= probabilities["BMI_Category"][1][row["BMI_Category"]]

    # Age_Category
    score_0 *= probabilities["Age_Category"][0][row["Age_Category"]]
    score_1 *= probabilities["Age_Category"][1][row["Age_Category"]]

    # PhysActivity
    score_0 *= probabilities["PhysActivity"][0][row["PhysActivity"]]
    score_1 *= probabilities["PhysActivity"][1][row["PhysActivity"]]

    # Predizione finale
    prediction = 0 if score_0 > score_1 else 1

    return prediction, score_0, score_1


In [71]:
predictions_v2 = training_df.apply(
    lambda row: predict_naive_bayes_v2(row)[0],
    axis=1
)

training_df["Predicted_v2"] = predictions_v2

predictions_v2.value_counts()

0    40226
1     9774
Name: count, dtype: int64

# Costruzione della nuova tabella dei risultati

In [72]:
results_v2 = pd.DataFrame({
    "Actual": training_df["Diabetes_binary"],
    "Predicted": predictions_v2
})

results_v2.head()

,Actual,Predicted
0,0,0
1,0,1
2,0,0
3,1,0
4,0,0


# Calcolo manuale della nuova Confusion Matrix

In [73]:
tp_v2 = len(
    training_df[(training_df["Diabetes_binary"] == 1) & (training_df["Predicted_v2"] == 1)]
)

tn_v2 = len(
    training_df[ (training_df["Diabetes_binary"] == 0) & (training_df["Predicted_v2"] == 0)]
)

fp_v2 = len(
    training_df[ (training_df["Diabetes_binary"] == 0) & (training_df["Predicted_v2"] == 1)]
)

fn_v2 = len(
    training_df[ (training_df["Diabetes_binary"] == 1) & (training_df["Predicted_v2"] == 0)]
)

print("TP =", tp_v2)
print("TN =", tn_v2)
print("FP =", fp_v2)
print("FN =", fn_v2)

# Visualizzazione della matrice

confusion_matrix_v2_df = pd.DataFrame(
    [
        [tn_v2, fp_v2],
        [fn_v2, tp_v2]
    ],
    columns=["Predicted 0", "Predicted 1"],
    index=["Actual 0", "Actual 1"]
)

confusion_matrix_v2_df

TP = 2774
TN = 36033
FP = 7000
FN = 4193


,Predicted 0,Predicted 1
Actual 0,36033,7000
Actual 1,4193,2774


# Ricalcolo Accuracy

In [74]:
accuracy_v2 = (tp_v2 + tn_v2) / (tp_v2 + tn_v2 + fp_v2 + fn_v2)

print("Accuracy Ricalcolata: " , accuracy_v2)

Accuracy Ricalcolata:  0.77614


# Ricalcolo Precision

In [75]:
precision_v2 = tp_v2 / (tp_v2 + fp_v2)

print("Precision Ricalcolata: ", precision_v2)

Precision Ricalcolata:  0.28381420094127274


# Ricalcolo Recall

In [76]:
recall_v2 = tp_v2 / (tp_v2 + fn_v2)

print("Recall Ricolacolata: ", recall_v2)

Recall Ricolacolata:  0.39816276733170664


# Ricalcolo F1-Score

In [77]:
f1_v2 = 2 * precision_v2 * recall_v2 / (precision_v2 + recall_v2)

print("F1-Score Ricalcolata: ", f1_v2)

F1-Score Ricalcolata:  0.33140194731497524


### Analisi critica dei risultati – Confronto tra modello originale e modello ottimizzato

Dopo aver applicato il classificatore Naive Bayes manuale al dataset `training.csv`, sono state calcolate le metriche di performance sia per il modello originale (basato sulle probabilità stimate da `manuale.csv`) sia per il modello ottimizzato (con probabilità a priori ricalcolate su un campione più ampio di 100 osservazioni).

| Metrica   | Modello originale | Modello ottimizzato |
|-----------|-------------------|---------------------|
| Accuracy  | 60.78%            | **77.61%**          |
| Precision | 22.28%            | **28.38%**          |
| Recall    | **72.96%**        | 39.82%              |
| F1-Score  | 34.14%            | **33.14%**          |

### Interpretazione del confronto

- **Accuracy**  
  L’accuracy aumenta in modo significativo (dal 60.78% al 77.61%).  
  Questo miglioramento è dovuto al fatto che le nuove probabilità a priori riflettono meglio la distribuzione reale delle classi nel dataset.

- **Precision**  
  La precision migliora dal 22.28% al 28.38%.  
  Il modello ottimizzato commette meno falsi positivi rispetto alla versione originale.

- **Recall**  
  La recall diminuisce in modo marcato, dal 72.96% al 39.82%.  
  Il modello diventa più “conservativo”: classifica come diabetici solo i casi più probabili, riducendo i falsi positivi ma aumentando i falsi negativi.

- **F1-Score**  
  Rimane sostanzialmente stabile (34.14% → 33.14%).  
  Questo indica che il miglioramento in precision è compensato dalla perdita in recall.

### Considerazioni complessive

L’ottimizzazione ha prodotto un modello:

- **più preciso** (meno falsi positivi),
- **più accurato** nel complesso,
- **ma meno sensibile** (individua meno casi di diabete).

In altre parole:

- Il modello originale era **molto sensibile ma poco preciso**.  
- Il modello ottimizzato è **più bilanciato**, ma perde capacità di individuare i casi positivi.

### Conclusione

L’ottimizzazione ha migliorato alcune metriche (Accuracy, Precision), ma ha peggiorato la Recall.  
Questo comportamento è coerente con il fatto che:

- le probabilità condizionate sono rimaste quelle del dataset molto piccolo (`manuale.csv`),
- solo le probabilità a priori sono state aggiornate,
- il modello ora riflette meglio la distribuzione reale delle classi, ma non ha informazioni migliori sulle relazioni tra feature e target.

Il risultato finale mostra che l’ottimizzazione parziale migliora la robustezza generale del modello, ma non può compensare la scarsa qualità delle probabilità condizionate.


Questa analisi costituisce la base per il confronto con il classificatore ad albero di decisione sviluppato dal compagno di gruppo,